In [1]:
import os

In [2]:
os.chdir("../")

In [3]:
%pwd

'd:\\Projects\\Kidney-Disease-Classification-Deep-Learning-Project'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    model_path: Path
    training_data: Path
    all_params: dict
    params_image_size: list
    params_batch_size: int

In [5]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):

        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)



    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        training = self.config.training
        training_data = self.config.data_transformation
        params = self.params

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            model_path=training.fine_tuned_model_path,
            training_data=training_data.root_dir,
            all_params=params,
            params_image_size=params.IMAGE_SIZE,
            params_batch_size=params.BATCH_SIZE
        )

        return model_evaluation_config

In [7]:
import keras
from cnnClassifier import logger

In [19]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config



    def _test_generator(self):
        datagen = keras.src.legacy.preprocessing.image.ImageDataGenerator(
            preprocessing_function=keras.applications.vgg16.preprocess_input
        )

        self.test_generator = datagen.flow_from_directory(
            os.path.join(self.config.training_data, "test"),
            target_size=self.config.params_image_size[:2],
            batch_size=self.config.params_batch_size,
            class_mode="categorical",
            shuffle=False
        )



    @staticmethod
    def load_model(path: Path) -> keras.Model:
        return keras.models.load_model(path)



    def evaluation(self):
        self.model = self.load_model(self.config.model_path)
        self._test_generator()
        self.score = self.model.evaluate(self.test_generator)



    def save_score(self):
        scores = {"loss": self.score[0], "accuracy":self.score[1]}
        save_json(path=Path(os.path.join(self.config.root_dir, "scores.json")), data=scores)

In [20]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.evaluation()
    model_evaluation.save_score()
except Exception as e:
    raise e

[2026-09-07 18:38:30,459: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-07 18:38:30,462: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-07 18:38:30,464: INFO: common: created directory at: artifacts/model_evaluation]
Found 1869 images belonging to 4 classes.
59/59 ━━━━━━━━━━━━━━━━━━━━ 295s 5s/step - accuracy: 0.9593 - loss: 0.2012
[2026-09-07 18:43:26,148: INFO: common: json file saved at: artifacts\model_evaluation\scores.json]
